# Qwen-3 0.6Bを継続事前学習をするサンプル

Qwen-3 0.6Bモデルを使用して、継続事前学習(Continued Pre-Training)を行うためのサンプルコードです。

Google Colab上での実行を想定していますが、ローカル環境でも動作します。Colabで実行するときに、インスタンスがbf16対応のGPUでない場合、より軽量なSmolLM2-360Mモデルを使用します。
※無料版の時はSmolLM2-360Mモデルを使用します。

In [1]:
import pandas as pd
from datetime import datetime
from datasets import Dataset

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)

In [2]:
cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
use_bf16 = cap[0] >= 8
dtype = torch.bfloat16 if use_bf16 else torch.float16
bf16 = use_bf16
fp16 = not use_bf16

# bf16: Brain Floating Point 16の略で、16ビットの浮動小数点数表現の一種。GPUメモリを削減するための低精度フォーマットに対応しているかを確認
# fp16: IEEE 754 半精度浮動小数点数 (Half-Precision Floating Point) の略で、16ビットの浮動小数点数表現の一種。主にグラフィックス処理や機械学習の分野で使用
print(f"CUDA Device Capability: {cap}, Using bf16: {bf16}, fp16: {fp16}")

CUDA Device Capability: (7, 5), Using bf16: False, fp16: True


In [3]:
# Colabの設定に応じてモデルを切り替え
if bf16:
    model_id = "Qwen/Qwen3-0.6B"
else:
    model_id = "HuggingFaceTB/SmolLM2-360M" # 無料版colabはGPUメモリが足りないためモデルを変える

print(f"Using model: {model_id}")

Using model: HuggingFaceTB/SmolLM2-360M


## データセットの準備

In [4]:
# サンプルデータセットを読み込み
url = "https://media.githubusercontent.com/media/SO0529/2601_software-design-chap3/refs/heads/main/abeja_common_crawl_0_sample.csv"
df = pd.read_csv(url)

print(df.head(5))

                                                 url  \
0                    https://cutout-persons.jp/item/   
1  https://www.uekikosho.com/news/%E5%8F%AF%E6%84...   
2           https://lovestrategy0011.com/line-id-440   
3  https://itevangelist.net/blog/tescom_tc391_tc315/   
4  https://bizenya.co.jp/products/tsubo-shiruko-g...   

                                             content  
0  ## ウェブでの作品製作で切っても切り離せない｢解像度｣\n## 同じ画像でも高いものと低い...  
1  ### 可愛らしくを心掛けた 動画「ルーアンの花市」\n植木良枝 花の刺繍画集「花の旅Ⅱ」に...  
2  目次\n### コンビニ店員にline(ライン)idを渡すよりも効率よくline(ライン)を...  
3  髪型を坊主頭にしてから10年程になり、自分の頭をバリカンで刈る事にも慣れました。\n私はバリ...  
4  「壷しるこ」の箱入りです。\n（配送時の商品保護の観点から、バラでのご注文はオンラインでは承...  


In [5]:
# content列だけを抽出してDataset化。サンプル実装なので1万件のみ使う
dataset = Dataset.from_pandas(df[["content"]]).select(range(10000))

# 学習に利用するためにcontentカラム名を "text" に統一してシャッフル
dataset = dataset.rename_column("content", "text")
dataset = dataset.shuffle(seed=42)

# train=9000件, valid=1000件 に分割
train_test = dataset.train_test_split(train_size=9000, test_size=1000, seed=42)

train_dataset = train_test["train"]
valid_dataset = train_test["test"]

## データセットをトークン化する

LLMに学習できるようにデータを全てトークン化する。

In [6]:

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if bf16 else torch.float32,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [7]:
# トークナイズ用の関数を定義
def tokenize_fn(batch):
    if model_id == "HuggingFaceTB/SmolLM2-360M":
        tokenizer.pad_token = tokenizer.eos_token
    # max_length 指定で pad、truncate
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=2048,  # 短め
        add_special_tokens=False,
    )

    # padは無視
    labels = []
    for ids in tokenized["input_ids"]:
        labels.append([
            (tok if tok != tokenizer.pad_token_id else -100)
            for tok in ids
        ])

    tokenized["labels"] = labels
    return tokenized

In [8]:
lm_train = train_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=train_dataset.column_names,
)

lm_valid = valid_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=valid_dataset.column_names,
)

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## 学習器の設定

In [ ]:
per_device_train_batch_size = 1 if fp16 else 2
gradient_accumulation_steps = 32  # 実効バッチを稼ぐ
learning_rate = 1e-6  # 元の重みを壊さないように小さくする
weight_decay = 0.001
warmup_ratio = 0.3
num_train_epochs = 10 if bf16 else 1  # 無料版は時間がかかるので1epoch

current_date = datetime.now().strftime("%Y%m%d")
output_dir_name = f"../results/continued-pretrain_{current_date}"

args = TrainingArguments(
    output_dir=output_dir_name,
    overwrite_output_dir=True,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    weight_decay=weight_decay,
    warmup_ratio=warmup_ratio,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,  # plot用でこまめに評価
    save_total_limit=1,
    bf16=bf16,
    fp16=fp16,
    report_to="tensorboard",
    dataloader_num_workers=2,
)

In [10]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    train_dataset=lm_train,
    eval_dataset=lm_valid,
)

/tmp/ipython-input-1047685771.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [ ]:
# NOTE: 約6時間程度かかる
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


In [ ]:
%load_ext tensorboard
%tensorboard --logdir {output_dir_name}